In [89]:
# Task 1: Data Understanding

In [90]:
# Import Required Libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [91]:
# Load dataset
df = pd.read_csv("credit_analysis_dataset_with_missing_outliers.csv")

# Display first few rows
df.head()

,CustomerID,CreditScore,Age,Income,LoanAmount,LoanDurationMonths,LoanPurpose,EmploymentStatus,DefaultStatus
0,C0001,488,78,195242,87146.0,46,Car Loan,Unemployed,False
1,C0002,791,34,98773,15554.0,32,Personal Loan,Unemployed,False
2,C0003,681,45,74846,57157.0,33,Personal Loan,Employed,False
3,C0004,325,19,169374,22051.0,51,Car Loan,Self-Employed,True
4,C0005,461,23,53171,21844.0,3,Education Loan,Self-Employed,False


In [92]:
# Dataset Dimensions and Structure

# Dataset shape
df.shape
# Dataset information
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   CustomerID          5000 non-null   object 
 1   CreditScore         5000 non-null   int64  
 2   Age                 5000 non-null   int64  
 3   Income              5000 non-null   int64  
 4   LoanAmount          4750 non-null   float64
 5   LoanDurationMonths  5000 non-null   int64  
 6   LoanPurpose         5000 non-null   object 
 7   EmploymentStatus    5000 non-null   object 
 8   DefaultStatus       5000 non-null   bool   
dtypes: bool(1), float64(1), int64(4), object(3)
memory usage: 317.5+ KB


In [93]:
# Check Missing Values

df.isnull().sum()

CustomerID              0
CreditScore             0
Age                     0
Income                  0
LoanAmount            250
LoanDurationMonths      0
LoanPurpose             0
EmploymentStatus        0
DefaultStatus           0
dtype: int64

In [94]:
# Handle Missing Values

# Fill missing LoanAmount values with median
df['LoanAmount'] = df['LoanAmount'].fillna(df['LoanAmount'].median())


# Verify missing values are handled
df.isnull().sum()

CustomerID            0
CreditScore           0
Age                   0
Income                0
LoanAmount            0
LoanDurationMonths    0
LoanPurpose           0
EmploymentStatus      0
DefaultStatus         0
dtype: int64

In [95]:
# Summary Statistics (Outlier Inspection)

# Summary statistics
df.describe()

,CreditScore,Age,Income,LoanAmount,LoanDurationMonths
count,5000.0000,5000.000000,5000.000000,5000.000000,5000.00000
mean,574.1878,49.108600,109882.112600,51524.990400,30.42400
std,157.5695,18.138032,51697.775394,27834.483792,17.48648
min,300.0000,18.000000,20068.000000,1013.000000,1.00000
25%,440.0000,34.000000,64618.500000,28117.000000,15.00000
50%,572.0000,49.000000,109910.000000,52272.500000,31.00000
75%,709.0000,65.000000,154108.000000,75070.000000,45.00000
max,850.0000,80.000000,199959.000000,99974.000000,180.00000


In [96]:
# Task 2: Data Modeling and Analysis

In [97]:
# Encoding Categorical Variables

# Convert categorical variables to dummy variables
df_encoded = pd.get_dummies(
    df,
    columns=['LoanPurpose', 'EmploymentStatus'],
    drop_first=True
)

In [98]:
# Define Features and Target Variable

X = df_encoded.drop(['CustomerID', 'DefaultStatus'], axis=1)
y = df_encoded['DefaultStatus']

In [99]:
# Train–Test Split (70% / 30%)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [100]:
# Feature Scaling

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [101]:
# Build Logistic Regression Model

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [102]:
# Make Predictions

y_pred = model.predict(X_test_scaled)

In [103]:
# Model Evaluation Metrics

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
accuracy
# Confusion Matrix
conf_matrix = confusion_matrix(y_test, y_pred)
conf_matrix
# Classification Report
class_report = classification_report(y_test, y_pred)
print(class_report)

              precision    recall  f1-score   support

       False       0.52      0.27      0.36       752
        True       0.50      0.74      0.60       748

    accuracy                           0.51      1500
   macro avg       0.51      0.51      0.48      1500
weighted avg       0.51      0.51      0.48      1500



In [104]:
# Task 3: Validation and Interpretation

In [105]:
# Validation Using Confusion Matrix and Classification Report

In [106]:
# Confusion matrix values
TN, FP, FN, TP = conf_matrix.ravel()
TN, FP, FN, TP

(np.int64(205), np.int64(547), np.int64(191), np.int64(557))

In [107]:
# Task 4: Insights and Recommendations

In [108]:
# Feature importance from logistic regression
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
})

feature_importance.sort_values(by='Coefficient', ascending=False)


,Feature,Coefficient
6,LoanPurpose_Home Loan,0.035622
8,EmploymentStatus_Self-Employed,0.021919
2,Income,0.014057
9,EmploymentStatus_Unemployed,0.014029
4,LoanDurationMonths,0.011374
7,LoanPurpose_Personal Loan,0.002709
5,LoanPurpose_Education Loan,-0.000003
1,Age,-0.016888
3,LoanAmount,-0.034023
0,CreditScore,-0.053115


In [109]:
# Task 5: Ethical and Responsible Analysis

In [110]:
# Check class distribution (imbalance)
y.value_counts(normalize=True)

DefaultStatus
True     0.5086
False    0.4914
Name: proportion, dtype: float64